# 05 - Gold merge

MERGEs gold-eligible jobs (India + English) into `gold.jobs` (SCD Type 1, one row per `job_key`), rebuilds `gold.job_skills` for the jobs that changed and recomputes `fit_score` against `ops.user_profile`.

| Rule | Behaviour |
|---|---|
| Source | `silver.jobs_clean` joined to `silver.jobs_enriched` on `job_key` **and** `description_hash` (enrichment must match the current description), `gold_eligible = true` |
| Insert | New `job_key` -> inserted with tracking defaults: `is_applied = false`, `is_hidden = false`, `application_status = 'not_applied'` |
| Update | Only when a pipeline column actually changed (description, enrichment, role, sightings...). **Tracking columns are never touched** |
| Delete | Never. A job that stops being eligible keeps its gold row, so your tracking is never lost |
| `gold.job_skills` | Deleted and re-inserted only for jobs inserted/updated in this run (`full_rebuild_skills` rebuilds everything) |
| Fit | `fit_*` columns recomputed for all jobs; written only when changed |

**Mark jobs as applied / hidden** directly in SQL:
```sql
UPDATE jobseeker.gold.jobs
SET is_applied = true, applied_at = current_timestamp(), application_status = 'applied', status_updated_at = current_timestamp()
WHERE job_key = '...';
```

**Edit your profile** (next run re-scores every job):
```sql
UPDATE jobseeker.ops.user_profile
SET skills = array('Databricks', 'Python', 'SQL', 'PySpark', 'Cloud', 'Azure', 'Azure Data Factory', 'Kafka'),
    preferred_cities = array('Bengaluru', 'Hyderabad'), updated_at = current_timestamp()
WHERE profile_id = 'default';
```

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.dropdown("full_rebuild_skills", "false", ["false", "true"], "Rebuild gold.job_skills for all jobs")
FULL_REBUILD_SKILLS = dbutils.widgets.get("full_rebuild_skills") == "true"

SILVER_TABLE = table("silver", "jobs_clean")
ENRICHED_TABLE = table("silver", "jobs_enriched")
GOLD_TABLE = table("gold", "jobs")
SKILLS_TABLE = table("gold", "job_skills")
REF_SKILLS_TABLE = table("ops", "ref_skills")
REF_ROLES_TABLE = table("ops", "ref_roles")
PROFILE_TABLE = table("ops", "user_profile")

# Pipeline-owned columns (refreshed by the MERGE). Everything else in gold.jobs is user-owned, lifecycle, fit or audit.
PIPELINE_COLUMNS = [
    "job_key", "source", "ats_type", "company_name", "company_url", "job_id", "title", "location", "posted_date", "posted_days",
    "job_url", "description", "seniority_level", "employment_type", "job_function", "industries",
    "experience_min_years", "experience_max_years", "experience_level", "skills", "skill_groups", "category", "category_score",
    "role_title", "role_score", "role_alternative", "role_method",
    "first_seen_at", "last_seen_at", "times_seen", "description_hash", "enrichment_version",
]
# Set only on INSERT, never overwritten by the MERGE: user tracking + lifecycle (lifecycle is managed by 06_cleanup)
INSERT_DEFAULTS = {
    "is_applied": "false",
    "applied_at": "CAST(NULL AS TIMESTAMP)",
    "is_hidden": "false",
    "hidden_at": "CAST(NULL AS TIMESTAMP)",
    "application_status": "'not_applied'",
    "status_updated_at": "CAST(NULL AS TIMESTAMP)",
    "is_active": "true",
    "expired_at": "CAST(NULL AS TIMESTAMP)",
}
# Columns added to gold.jobs after its first version (01_setup adds them too; this guards a skipped setup run)
COLUMN_TYPES = {
    "is_applied": "BOOLEAN", "applied_at": "TIMESTAMP", "is_hidden": "BOOLEAN", "hidden_at": "TIMESTAMP",
    "application_status": "STRING", "status_updated_at": "TIMESTAMP", "is_active": "BOOLEAN", "expired_at": "TIMESTAMP",
    "role_title": "STRING", "role_score": "DOUBLE", "role_alternative": "STRING", "role_method": "STRING",
    "fit_score": "INT", "fit_role": "DOUBLE", "fit_skills": "DOUBLE", "fit_experience": "DOUBLE", "fit_location": "DOUBLE",
    "fit_matched_skills": "ARRAY<STRING>", "fit_scored_at": "TIMESTAMP",
}

gold_columns = {field.name for field in spark.table(GOLD_TABLE).schema.fields}
missing_columns = [f"{name} {data_type}" for name, data_type in COLUMN_TYPES.items() if name not in gold_columns]
if missing_columns:
    spark.sql(f"ALTER TABLE {GOLD_TABLE} ADD COLUMNS ({', '.join(missing_columns)})")
    print(f"Added columns to {GOLD_TABLE}: {missing_columns}")

enriched_columns = {field.name for field in spark.table(ENRICHED_TABLE).schema.fields}
missing_enriched = [name for name in ("role_title", "role_score", "role_alternative", "role_method") if name not in enriched_columns]
if missing_enriched:
    raise RuntimeError(f"{ENRICHED_TABLE} lacks {missing_enriched}: run 01_setup and 04_enrich first")

print(f"target={GOLD_TABLE} full_rebuild_skills={FULL_REBUILD_SKILLS}")

## Build the eligible source set

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW gold_updates AS
SELECT
  s.job_key, s.source, s.ats_type, s.company_name, s.company_url, s.job_id, s.title, s.location,
  s.posted_date, s.posted_days, s.job_url, s.description,
  s.seniority_level, s.employment_type, s.job_function, s.industries,
  e.experience_min_years, e.experience_max_years, e.experience_level,
  e.skills, e.skill_groups, e.category, e.category_score,
  e.role_title, e.role_score, e.role_alternative, e.role_method,
  s.first_seen_at, s.last_seen_at, s.times_seen, s.description_hash, e.enrichment_version
FROM {SILVER_TABLE} s
JOIN {ENRICHED_TABLE} e
  ON e.job_key = s.job_key AND e.description_hash = s.description_hash
WHERE e.gold_eligible
""")

eligible_count = spark.table("gold_updates").count()
duplicate_keys = spark.table("gold_updates").groupBy("job_key").count().where("count > 1").count()
if duplicate_keys:
    raise ValueError(f"{duplicate_keys} duplicate job_key values in the gold source; silver/enriched must be unique per job_key")
without_role = spark.table("gold_updates").where("role_title IS NULL").count()
print(f"eligible jobs: {eligible_count} (without a role yet: {without_role} - 04_enrich assigns them in later runs)")

## MERGE into gold.jobs

In [ ]:
with task_run("gold_merge") as metrics:
    refresh_columns = [column for column in PIPELINE_COLUMNS if column != "job_key"]
    changed = " OR ".join(f"NOT (t.{column} <=> s.{column})" for column in refresh_columns)
    update_set = ", ".join(f"t.{column} = s.{column}" for column in refresh_columns)

    insert_columns = PIPELINE_COLUMNS + list(INSERT_DEFAULTS) + ["gold_inserted_at", "gold_updated_at", "_job_run_id"]
    insert_values = (
        [f"s.{column}" for column in PIPELINE_COLUMNS]
        + list(INSERT_DEFAULTS.values())
        + ["current_timestamp()", "current_timestamp()", f"'{JOB_RUN_ID}'"]
    )

    merge_result = spark.sql(f"""
        MERGE INTO {GOLD_TABLE} t
        USING gold_updates s
        ON t.job_key = s.job_key
        WHEN MATCHED AND ({changed}) THEN UPDATE SET
          {update_set},
          t.gold_updated_at = current_timestamp(),
          t._job_run_id = '{JOB_RUN_ID}'
        WHEN NOT MATCHED THEN INSERT ({", ".join(insert_columns)})
          VALUES ({", ".join(insert_values)})
    """).first().asDict()

    inserted = int(merge_result.get("num_inserted_rows", 0))
    updated = int(merge_result.get("num_updated_rows", 0))

    # Rebuild job_skills for jobs touched in this run (or everything on request)
    touched_filter = "TRUE" if FULL_REBUILD_SKILLS else f"_job_run_id = '{JOB_RUN_ID}'"
    if FULL_REBUILD_SKILLS:
        spark.sql(f"TRUNCATE TABLE {SKILLS_TABLE}")
    else:
        spark.sql(f"DELETE FROM {SKILLS_TABLE} WHERE job_key IN (SELECT job_key FROM {GOLD_TABLE} WHERE {touched_filter})")

    spark.sql(f"""
        INSERT INTO {SKILLS_TABLE} (job_key, skill, skill_group, category, company_name, posted_date, _updated_at)
        SELECT g.job_key, g.skill, coalesce(r.skill_group, 'Other') AS skill_group, g.category, g.company_name, g.posted_date,
               current_timestamp()
        FROM (
          SELECT job_key, category, company_name, posted_date, explode(skills) AS skill
          FROM {GOLD_TABLE}
          WHERE {touched_filter}
        ) g
        LEFT JOIN (SELECT DISTINCT skill, skill_group FROM {REF_SKILLS_TABLE}) r ON r.skill = g.skill
    """)
    skill_rows = spark.sql(
        f"SELECT count(*) AS n FROM {SKILLS_TABLE} WHERE job_key IN (SELECT job_key FROM {GOLD_TABLE} WHERE {touched_filter})"
    ).first()["n"]

    metrics.update(rows_in=eligible_count, rows_out=inserted + updated, rows_rejected=0)
    metrics["details"].update(inserted=inserted, updated=updated, unchanged=eligible_count - inserted - updated,
                              job_skills_rows=int(skill_rows), full_rebuild_skills=FULL_REBUILD_SKILLS)
    print(f"inserted={inserted} updated={updated} unchanged={eligible_count - inserted - updated} job_skills_rows={skill_rows}")

try:
    dbutils.jobs.taskValues.set(key="gold_changed", value=inserted + updated)
except Exception:
    pass  # not running inside a job

## Fit score
Recomputed for every gold job on each run from `ops.user_profile`, so editing your profile re-ranks everything on the next run. Only rows whose score changed are written.

| Component | Rule |
|---|---|
| `fit_role` | `role_title` is target role #i -> `1 - 0.05*i` (min 0.7); `role_alternative` is a target -> 0.5; same category as a target role -> 0.3; else 0 |
| `fit_skills` | Profile skills found in the job / min(#profile skills, 5), capped at 1. A profile skill that is a skill group name (e.g. `Cloud`) matches any skill of that group |
| `fit_experience` | Job minimum unknown -> 0.7; job minimum <= your `max_years` -> 1; else minus 0.25 per extra year |
| `fit_location` | No preferred cities, or a preferred city in the location -> 1; else 0.5 |

`fit_score = round(100 * weighted average)` using the profile weights.

In [ ]:
with task_run("gold_fit") as metrics:
    profile_rows = spark.table(PROFILE_TABLE).orderBy(F.col("updated_at").desc_nulls_last()).limit(1).collect()
    if not profile_rows:
        raise RuntimeError(f"{PROFILE_TABLE} is empty - run 01_setup to seed it")
    profile = profile_rows[0]

    # Target roles -> rank and categories
    role_category = {row["role_title"]: row["category"] for row in spark.table(REF_ROLES_TABLE).select("role_title", "category").collect()}
    unknown_roles = [role for role in profile["target_roles"] or [] if role not in role_category]
    if unknown_roles:
        print(f"WARNING: target roles not in {REF_ROLES_TABLE} (ignored): {unknown_roles}")
    target_roles = [role for role in profile["target_roles"] or [] if role in role_category]
    role_rank = {role: rank for rank, role in reversed(list(enumerate(target_roles)))}
    target_categories = {role_category[role] for role in target_roles}

    # Profile skills -> skill group (e.g. "Cloud") or canonical skill through ref_skills aliases
    alias_to_skill, group_by_lower = {}, {}
    for row in spark.table(REF_SKILLS_TABLE).select("skill", "skill_group", "aliases").collect():
        alias_to_skill.setdefault(row["skill"].lower(), row["skill"])
        for alias in row["aliases"] or []:
            alias_to_skill.setdefault(alias.lower(), row["skill"])
        if row["skill_group"]:
            group_by_lower[row["skill_group"].lower()] = row["skill_group"]
    profile_skills = {}  # label -> (kind, value)
    for raw in profile["skills"] or []:
        key = raw.strip().lower()
        if key in group_by_lower:
            profile_skills.setdefault(group_by_lower[key], ("group", group_by_lower[key]))
        elif key in alias_to_skill:
            profile_skills.setdefault(alias_to_skill[key], ("skill", alias_to_skill[key].lower()))
        elif key:
            print(f"WARNING: profile skill '{raw}' is not in {REF_SKILLS_TABLE}; matched literally")
            profile_skills.setdefault(raw.strip(), ("skill", key))
    skill_denominator = max(1, min(len(profile_skills), 5))

    user_max_years = profile["max_years"]
    cities = [city.strip().lower() for city in profile["preferred_cities"] or [] if city and city.strip()]
    weights = {name: float(profile[f"weight_{name}"] or 0.0) for name in ("role", "skills", "experience", "location")}
    total_weight = sum(weights.values())
    if total_weight <= 0:
        raise ValueError(f"{PROFILE_TABLE} weights must add up to more than 0: {weights}")

    def fit(row):
        if row["role_title"] in role_rank:
            fit_role = max(0.7, 1.0 - 0.05 * role_rank[row["role_title"]])
        elif row["role_alternative"] in role_rank:
            fit_role = 0.5
        elif row["category"] in target_categories:
            fit_role = 0.3
        else:
            fit_role = 0.0

        job_skills = {skill.lower() for skill in row["skills"] or []}
        job_groups = set(row["skill_groups"] or [])
        matched = sorted(
            label for label, (kind, value) in profile_skills.items()
            if (value in job_groups if kind == "group" else value in job_skills)
        )
        fit_skills = min(1.0, len(matched) / skill_denominator)

        job_min_years = row["experience_min_years"]
        if job_min_years is None:
            fit_experience = 0.7
        elif user_max_years is None or job_min_years <= user_max_years:
            fit_experience = 1.0
        else:
            fit_experience = max(0.0, 1.0 - 0.25 * (job_min_years - user_max_years))

        location = (row["location"] or "").lower()
        fit_location = 1.0 if not cities or any(city in location for city in cities) else 0.5

        components = {"role": fit_role, "skills": fit_skills, "experience": fit_experience, "location": fit_location}
        score = int(round(100 * sum(weights[name] * value for name, value in components.items()) / total_weight))
        return (row["job_key"], score, round(fit_role, 3), round(fit_skills, 3), round(fit_experience, 3), round(fit_location, 3), matched)

    gold_rows = (
        spark.table(GOLD_TABLE)
        .select("job_key", "role_title", "role_alternative", "category", "skills", "skill_groups", "experience_min_years", "location")
        .collect()
    )
    fit_rows = [fit(row) for row in gold_rows]

    fit_updated = 0
    if fit_rows:
        (
            spark.createDataFrame(
                fit_rows,
                "job_key STRING, fit_score INT, fit_role DOUBLE, fit_skills DOUBLE, fit_experience DOUBLE, fit_location DOUBLE, "
                "fit_matched_skills ARRAY<STRING>",
            )
            .createOrReplaceTempView("fit_updates")
        )
        fit_columns = ["fit_score", "fit_role", "fit_skills", "fit_experience", "fit_location", "fit_matched_skills"]
        fit_result = spark.sql(f"""
            MERGE INTO {GOLD_TABLE} t
            USING fit_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED AND ({" OR ".join(f"NOT (t.{column} <=> s.{column})" for column in fit_columns)}) THEN UPDATE SET
              {", ".join(f"t.{column} = s.{column}" for column in fit_columns)},
              t.fit_scored_at = current_timestamp()
        """).first().asDict()
        fit_updated = int(fit_result.get("num_updated_rows", 0))

    metrics.update(rows_in=len(gold_rows), rows_out=fit_updated, rows_rejected=0)
    metrics["details"].update(fit_updated=fit_updated, target_roles=len(target_roles), profile_skills=sorted(profile_skills),
                              unknown_roles=unknown_roles)
    print(f"fit scored={len(fit_rows)} changed={fit_updated} | target roles={len(target_roles)} profile skills={sorted(profile_skills)}")

## Inspect

In [ ]:
gold = spark.table(GOLD_TABLE)
display(
    gold.groupBy("source")
    .agg(
        F.count("*").alias("jobs"),
        F.sum(F.col("is_applied").cast("int")).alias("applied"),
        F.sum(F.col("is_hidden").cast("int")).alias("hidden"),
        F.round(F.avg(F.size("skills")), 1).alias("avg_skills"),
        F.sum(F.col("experience_min_years").isNotNull().cast("int")).alias("with_experience"),
        F.max("last_seen_at").alias("latest_seen"),
    )
    .orderBy("source")
)

In [ ]:
display(gold.groupBy("category").count().orderBy(F.col("count").desc()))

In [ ]:
display(gold.groupBy("experience_level").count().orderBy(F.col("count").desc()))

In [ ]:
display(
    spark.table(SKILLS_TABLE).groupBy("skill", "skill_group").agg(F.countDistinct("job_key").alias("jobs"))
    .orderBy(F.col("jobs").desc()).limit(30)
)

In [ ]:
gold = spark.table(GOLD_TABLE)
display(gold.groupBy("category", "role_title", "role_method").count().orderBy(F.col("count").desc()).limit(50))
display(
    gold.where("NOT coalesce(is_hidden, false) AND coalesce(is_active, true) AND NOT coalesce(is_applied, false)")
    .select("fit_score", "posted_date", "company_name", "title", "role_title", "role_score", "location", "experience_min_years",
            "fit_role", "fit_skills", "fit_experience", "fit_location", "fit_matched_skills", "job_url", "job_key")
    .orderBy(F.col("fit_score").desc_nulls_last(), F.col("posted_date").desc_nulls_last())
    .limit(50)
)

In [ ]:
display(spark.table(table("ops", "pipeline_runs")).orderBy(F.col("started_at").desc()).limit(10))